# Apache Iceberg Lakehouse — Student Notebook

1. **Setup** — configuration, S3 Tables catalog, upload scripts and data
2. **Bronze** — run Glue jobs, validate with Athena
3. **Silver** — run Glue job, validate with Athena
4. **Gold** — run Glue job, validate with Athena
5. **Iceberg Features** — snapshots, time-travel

---

# Part 1: Setup

## Step 1: Configuration

Set up all resource names matching the CloudFormation stack outputs. Bronze uses a
standard S3 bucket with Glue Catalog Iceberg tables. Silver and Gold use **S3 Tables**
(managed Iceberg) accessed through the `s3tablescatalog` federated catalog.

Fill in the values from your CloudFormation stack outputs.

In [3]:
import os, time
import boto3
import pandas as pd

AWS_REGION = 'us-east-1'

session = boto3.Session()
print("boto3 credentials found:", session.get_credentials() is not None)
print("region:", session.region_name)

ACCOUNT_ID = 'YOUR_ACCOUNT_ID'  # replace with your 12-digit AWS account ID
BUCKET = f'ecommerce-analytics-datalake-dev-{ACCOUNT_ID}'  # DataLakeBucketName
DATABASE = 'ecommerce_analytics_bronze_dev'
ATHENA_WORKGROUP = 'ecommerce-analytics-workgroup-dev'      # AthenaWorkgroupName output

# S3 Tables catalog (silver + gold layers)
TABLE_BUCKET_NAME = f'ecommerce-analytics-gold-dev-{ACCOUNT_ID}'  # from GoldTableBucketARN (bucket name)
S3T_CATALOG = f's3tablescatalog/{TABLE_BUCKET_NAME}'
SILVER_DB = 'ecommerce_silver'
GOLD_DB = 'ecommerce_gold'

# Glue job names
CDC_JOB = 'ecommerce-analytics-cdc-bronze-dev'
EVENTS_JOB = 'ecommerce-analytics-events-bronze-dev'
SILVER_JOB = 'ecommerce-analytics-silver-layer-dev'
GOLD_JOB = 'ecommerce-analytics-gold-layer-dev'

s3 = boto3.client('s3', region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)
athena = boto3.client('athena', region_name=AWS_REGION)

print('\u2705 Configuration set')

boto3 credentials found: True
region: us-east-1
✅ Configuration set


## Step 2: Register S3 Tables Catalog (one-time after stack deploy)

The Silver and Gold layers write to **S3 Tables** (managed Iceberg). To query these
tables from Athena and allow Glue jobs to discover them, we must register a
`s3tablescatalog` federated catalog in the Glue Data Catalog. This bridges the
S3 Tables API with AWS analytics services (Athena, Glue, EMR). The catalog uses
`IAM_ALLOWED_PRINCIPALS` so access is controlled by IAM policies, not Lake Formation grants.

This only needs to run once per account/region — it's safe to re-run (idempotent).

In [4]:
try:
    glue.create_catalog(
        Name='s3tablescatalog',
        CatalogInput={
            'FederatedCatalog': {
                'Identifier': f'arn:aws:s3tables:{AWS_REGION}:{ACCOUNT_ID}:bucket/*',
                'ConnectionName': 'aws:s3tables'
            },
            'CreateDatabaseDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'CreateTableDefaultPermissions': [
                {'Principal': {'DataLakePrincipalIdentifier': 'IAM_ALLOWED_PRINCIPALS'}, 'Permissions': ['ALL']}
            ],
            'AllowFullTableExternalDataAccess': 'True'
        }
    )
    print('\u2705 s3tablescatalog registered')
except glue.exceptions.AlreadyExistsException:
    print('\u2705 s3tablescatalog already exists')

✅ s3tablescatalog already exists


## Step 3: Upload Glue Scripts and Datasets

Upload the four Glue ETL scripts and the CDC/events datasets to S3. The Glue jobs
fetch their scripts from `s3://{bucket}/glue-scripts/` at runtime, so this must
happen before running any jobs.

In [23]:
import glob as globmod

# Upload Glue scripts
scripts = {
    'glue-jobs/cdc-batch/src/iceberg-cdc-bronze-layer.py': 'glue-scripts/iceberg-cdc-bronze-layer.py',
    'glue-jobs/events/src/iceberg-events-bronze-layer.py': 'glue-scripts/iceberg-events-bronze-layer.py',
    'glue-jobs/silver/src/iceberg-silver-layer.py': 'glue-scripts/iceberg-silver-layer.py',
    'glue-jobs/gold/src/iceberg-gold-layer.py': 'glue-scripts/iceberg-gold-layer.py',
}
for local, s3_key in scripts.items():
    s3.upload_file(local, BUCKET, s3_key)
    print(f'\u2705 Uploaded {s3_key}')

# Upload datasets
for local_path in globmod.glob('datasets/**/*', recursive=True):
    if os.path.isfile(local_path):
        s3.upload_file(local_path, BUCKET, local_path)

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix='datasets/', MaxKeys=5)
for obj in resp.get('Contents', []):
    print(f'  {obj["Key"]} ({obj["Size"]:,} bytes)')
print('\u2705 Datasets uploaded')

✅ Uploaded glue-scripts/iceberg-cdc-bronze-layer.py
✅ Uploaded glue-scripts/iceberg-events-bronze-layer.py
✅ Uploaded glue-scripts/iceberg-silver-layer.py
✅ Uploaded glue-scripts/iceberg-gold-layer.py
  datasets/cdc_data/small/year=2025/table_name=orders/month=01/data_89a7daa5.json (154,082 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=02/data_45393fc2.json (178,709 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=03/data_d44f1e66.json (179,452 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=04/data_1c051040.json (177,904 bytes)
  datasets/cdc_data/small/year=2025/table_name=orders/month=05/data_88f6016a.json (177,918 bytes)
✅ Datasets uploaded


## Step 4: Helper Functions

`run_glue_job` starts a Glue job and polls until completion. `run_query` executes
Athena SQL and returns a DataFrame. The optional `catalog` and `database` parameters
let us query S3 Tables (silver/gold) by passing the federated catalog name.

In [8]:
def run_glue_job(job_name):
    run = glue.start_job_run(JobName=job_name)
    run_id = run['JobRunId']
    print(f'\u25b6 Started {job_name} (run: {run_id})')
    while True:
        status = glue.get_job_run(JobName=job_name, RunId=run_id)['JobRun']
        state = status['JobRunState']
        if state in ('SUCCEEDED', 'FAILED', 'STOPPED', 'ERROR', 'TIMEOUT'):
            break
        print(f'  {state}...', end='\r')
        time.sleep(30)
    if state == 'SUCCEEDED':
        print(f'\u2705 {job_name} SUCCEEDED ({status.get("ExecutionTime", 0)}s)')
    else:
        print(f'\u274c {job_name} {state}: {status.get("ErrorMessage", "")}')
    return state

def run_query(sql, catalog=None, database=None):
    params = {
        'QueryString': sql,
        'WorkGroup': ATHENA_WORKGROUP,
        'ResultConfiguration': {'OutputLocation': f's3://{BUCKET}/athena-results/'}
    }
    if catalog or database:
        params['QueryExecutionContext'] = {}
        if catalog:
            params['QueryExecutionContext']['Catalog'] = catalog
        if database:
            params['QueryExecutionContext']['Database'] = database
    exe = athena.start_query_execution(**params)
    qid = exe['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(1)
    if state != 'SUCCEEDED':
        raise Exception(status['QueryExecution']['Status'].get('StateChangeReason', state))
    results = athena.get_query_results(QueryExecutionId=qid)
    cols = [c['Label'] for c in results['ResultSet']['ResultSetMetadata']['ColumnInfo']]
    rows = [[f['VarCharValue'] if 'VarCharValue' in f else None for f in r['Data']] for r in results['ResultSet']['Rows'][1:]]
    return pd.DataFrame(rows, columns=cols)

print('\u2705 Helpers ready')

✅ Helpers ready


---
# Part 2: Bronze Layer

## Step 5: Run Bronze Glue Jobs

Two jobs ingest raw data into Iceberg tables on S3:
- **CDC Bronze** — reads Debezium CDC JSON, deduplicates by primary key, and uses
  `MERGE INTO` to upsert orders, users, and products.
- **Events Bronze** — reads clickstream JSON and appends to the events table.

In [9]:
run_glue_job(CDC_JOB)
run_glue_job(EVENTS_JOB)

▶ Started ecommerce-analytics-cdc-bronze-dev (run: jr_02b99babd42cc6637dc6956e4e87ab0c5b1ffb2344181272130d95253991d1e2)
✅ ecommerce-analytics-cdc-bronze-dev SUCCEEDED (135s)
▶ Started ecommerce-analytics-events-bronze-dev (run: jr_9b63fd5dfbec3c5cdd4691b64996e1dac74aa2a6a3175e91f59fd12c13aad0eb)
✅ ecommerce-analytics-events-bronze-dev SUCCEEDED (98s)


'SUCCEEDED'

## Step 6: Validate Bronze Layer

Write Athena queries to validate the bronze tables:
1. Count rows in each table (orders, users, products, events)
2. Check for duplicate order IDs (should be 0 after MERGE dedup)
3. Show event type distribution

Use `run_query()` — bronze tables are in the default Glue catalog, so no catalog parameter needed.
Reference tables as `{DATABASE}.table_name`.

In [10]:
# Write a UNION ALL query to count rows in each bronze table
bronze_counts = run_query(f"""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM {DATABASE}.orders
    UNION ALL
    SELECT 'users', COUNT(*) FROM {DATABASE}.users
    UNION ALL
    SELECT 'products', COUNT(*) FROM {DATABASE}.products
    UNION ALL
    SELECT 'events', COUNT(*) FROM {DATABASE}.events
""")
print(bronze_counts.to_string(index=False))

table_name row_count
     users       912
  products       391
    events       498
    orders      2133


In [11]:
# Check for duplicate order_ids
dupes = run_query(f"SELECT order_id, COUNT(*) AS c FROM {DATABASE}.orders GROUP BY order_id HAVING COUNT(*) > 1")
print(f'Duplicate orders: {len(dupes)}')

Duplicate orders: 0


In [12]:
# Show event type breakdown
event_types = run_query(f"SELECT event_type, COUNT(*) AS cnt FROM {DATABASE}.events GROUP BY event_type ORDER BY cnt DESC")
print(event_types.to_string(index=False))

 event_type cnt
  page_view 208
add_to_cart 120
   purchase  93
      click  77


---
# Part 3: Silver Layer

## Step 7: Run Silver Glue Job

Reads from Bronze (Glue Catalog / S3) and writes enriched tables to **S3 Tables**:
- **order_details** — orders joined with users, adds customer segments and lifetime value
- **enriched_events** — events joined with users and products, adds revenue and session sequence
- **product_performance** — aggregated metrics per product (views, carts, purchases, conversion rates)

In [14]:
run_glue_job(SILVER_JOB)

▶ Started ecommerce-analytics-silver-layer-dev (run: jr_d6779eac386b9ad217249d2f16a464aaf11fb8fe86f5c3068508f323f1a6c557)
✅ ecommerce-analytics-silver-layer-dev SUCCEEDED (161s)


'SUCCEEDED'

## Step 8: Validate Silver Layer

Write Athena queries to validate the silver tables:
1. Count rows in each table (order_details, enriched_events, product_performance)
2. Show customer segment distribution from order_details
3. Show product funnel conversion rates from product_performance

Silver tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=SILVER_DB`
to `run_query()`. Use unqualified table names (no database prefix in SQL).

In [15]:
#  Count rows in each silver table
silver_counts = run_query("""
    SELECT 'order_details' AS table_name, COUNT(*) AS row_count FROM order_details
    UNION ALL
    SELECT 'enriched_events', COUNT(*) FROM enriched_events
    UNION ALL
    SELECT 'product_performance', COUNT(*) FROM product_performance
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(silver_counts.to_string(index=False))

         table_name row_count
product_performance       391
      order_details      2024
    enriched_events       498


In [17]:
#  Show customer segment distribution (customer_segment, count, avg order value)
segments = run_query("""
    SELECT customer_segment,
           COUNT(*) AS orders,
           ROUND(AVG(total_amount), 2) AS avg_order_value
    FROM order_details
    GROUP BY customer_segment
    ORDER BY orders DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(segments.to_string(index=False))

customer_segment orders avg_order_value
           Loyal   1076          261.31
          Repeat    516          258.75
             VIP    217          262.23
             New    215          265.97


In [18]:
# : Show product funnel (category, page_views, purchases, conversion rate)
funnel = run_query("""
    SELECT category,
           SUM(page_views) AS page_views,
           SUM(purchases) AS purchases,
           ROUND(CASE WHEN SUM(page_views) = 0 THEN 0.0
                      ELSE CAST(SUM(purchases) AS DOUBLE) / SUM(page_views) END, 4) AS conversion_rate
    FROM product_performance
    GROUP BY category
    ORDER BY page_views DESC
""", catalog=S3T_CATALOG, database=SILVER_DB)
print(funnel.to_string(index=False))

     category page_views purchases conversion_rate
       Sports         60        24             0.4
  Electronics         18         5          0.2778
        Books         18         8          0.4444
Home & Garden          0         0             0.0
     Clothing          0         0             0.0


---
# Part 4: Gold Layer

## Step 9: Run Gold Glue Job

Reads from Silver (S3 Tables) and writes business-ready aggregations to **S3 Tables**:
- **customer_analytics** — monthly customer metrics (revenue, segments, geography)
- **realtime_metrics** — hourly event metrics (traffic sources, conversions, revenue)

In [24]:
run_glue_job(GOLD_JOB)

▶ Started ecommerce-analytics-gold-layer-dev (run: jr_0fdba6fc62378c071fae2ee3ca829d2dc9ece199fa88f7c2dee0507277dcff63)
✅ ecommerce-analytics-gold-layer-dev SUCCEEDED (89s)


'SUCCEEDED'

## Step 10: Validate Gold Layer

Write Athena queries to validate the gold tables:
1. Show monthly customer analytics (analysis_month, total_customers, total_revenue, avg_order_value)
2. Show recent hourly metrics (metric_hour, total_events, unique_users, hourly_revenue)

Gold tables are on **S3 Tables** — pass `catalog=S3T_CATALOG, database=GOLD_DB`.

In [25]:
monthly = run_query("""
    SELECT analysis_month, total_customers,
           ROUND(total_revenue, 2) AS total_revenue,
           ROUND(avg_order_value, 2) AS avg_order_value
    FROM customer_analytics
    ORDER BY analysis_month
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(monthly.to_string(index=False))
print(f'Months: {len(monthly)}')


                analysis_month total_customers total_revenue avg_order_value
2024-12-01 00:00:00.000000 UTC              35      32449.01      292.330000
2025-01-01 00:00:00.000000 UTC              82      37326.43      259.210000
2025-02-01 00:00:00.000000 UTC             103      39507.88      251.640000
2025-03-01 00:00:00.000000 UTC             128      43828.48      259.340000
2025-04-01 00:00:00.000000 UTC             132      45978.67      267.320000
2025-05-01 00:00:00.000000 UTC             139      40413.00      246.420000
2025-06-01 00:00:00.000000 UTC             151      46707.47      262.400000
2025-07-01 00:00:00.000000 UTC             153      43597.24      247.710000
2025-08-01 00:00:00.000000 UTC             140      44307.84      275.200000
2025-09-01 00:00:00.000000 UTC             159      48658.60      263.020000
2025-10-01 00:00:00.000000 UTC             162      49138.31      276.060000
2025-11-01 00:00:00.000000 UTC             153      43449.34      257.100000

In [26]:
hourly = run_query("""
    SELECT metric_hour, total_events, unique_users,
           ROUND(hourly_revenue, 2) AS hourly_revenue
    FROM realtime_metrics
    WHERE hourly_revenue > 0
    ORDER BY metric_hour DESC
    LIMIT 10
""", catalog=S3T_CATALOG, database=GOLD_DB)
print(hourly.to_string(index=False))


                   metric_hour total_events unique_users hourly_revenue
2026-09-30 07:00:00.000000 UTC            1            1         750.55
2026-09-27 14:00:00.000000 UTC            2            2         292.03
2026-09-20 03:00:00.000000 UTC            1            1         942.14
2026-09-18 13:00:00.000000 UTC            1            1         179.67
2026-09-12 04:00:00.000000 UTC            1            1         942.14
2026-09-09 16:00:00.000000 UTC            1            1         710.18
2026-09-05 10:00:00.000000 UTC            1            1          794.1
2026-09-05 04:00:00.000000 UTC            1            1         942.14
2026-09-02 14:00:00.000000 UTC            1            1         701.93
2026-08-24 07:00:00.000000 UTC            1            1         137.83


## Step 10b: Supplemental CDC Load

To demonstrate Iceberg snapshots and time travel, we apply the supplemental CDC records
(3 updates and 2 deletes on `orders`) to the bronze `orders` table.

The CDC job reads from `{path}/year={year}/table_name={table}/`, so the supplemental file is first
copied (server-side, inside S3) into that layout under a separate prefix. The CDC job is then
started with its input path overridden to that prefix. The `MERGE INTO` creates a second Iceberg
snapshot on the orders table, while the first snapshot is preserved for time travel.

In [29]:
# Copy the supplemental file into the year=/table_name= layout the CDC job expects
SUPP_SRC_KEY  = 'datasets/cdc_data/supplemental/table_name=orders/supplemental_cdc.json'
SUPP_PREFIX   = 'datasets/cdc_supplemental_run'
SUPP_DEST_KEY = f'{SUPP_PREFIX}/year=2025/table_name=orders/supplemental_cdc.json'

s3.copy_object(Bucket=BUCKET, Key=SUPP_DEST_KEY, CopySource={'Bucket': BUCKET, 'Key': SUPP_SRC_KEY})
print(f'✅ Copied to s3://{BUCKET}/{SUPP_DEST_KEY}')

# Run the CDC bronze job against the supplemental data (overrides the input path)
run = glue.start_job_run(
    JobName=CDC_JOB,
    Arguments={'--cdc_data_path': f's3://{BUCKET}/{SUPP_PREFIX}', '--target_year': '2025'}
)
run_id = run['JobRunId']
print(f'▶ Started {CDC_JOB} on supplemental data (run: {run_id})')

while True:
    state = glue.get_job_run(JobName=CDC_JOB, RunId=run_id)['JobRun']['JobRunState']
    if state in ('SUCCEEDED', 'FAILED', 'STOPPED', 'ERROR', 'TIMEOUT'):
        break
    time.sleep(30)
print(f'{"✅" if state == "SUCCEEDED" else "❌"} {CDC_JOB} {state}')

# Confirm the effect on the orders table
after = run_query(f"SELECT COUNT(*) AS row_count FROM {DATABASE}.orders")
print(f"orders rows after supplemental load: {after['row_count'][0]}  (was 2133)")

✅ Copied to s3://ecommerce-analytics-datalake-dev-YOUR_ACCOUNT_ID/datasets/cdc_supplemental_run/year=2025/table_name=orders/supplemental_cdc.json
▶ Started ecommerce-analytics-cdc-bronze-dev on supplemental data (run: jr_4805fc8e89439054d7a6e5cab89e99ec0832d1245e23b9833d590606cf780614)
✅ ecommerce-analytics-cdc-bronze-dev SUCCEEDED
orders rows after supplemental load: 2131  (was 2133)


---
# Part 5: Iceberg Features

## Step 11: Snapshot Inspection

Iceberg maintains a log of every change as immutable **snapshots**. Query the
`$snapshots` and `$history` metadata tables to see when data was written,
what operation occurred (append, overwrite), and how many files were added.

Use the bronze orders table for this — reference metadata tables as:
- `"{DATABASE}"."orders$snapshots"`
- `"{DATABASE}"."orders$history"`

In [30]:
# Show: snapshot_id, committed_at, operation, summary
snapshots = run_query(f"""
    SELECT snapshot_id, committed_at, operation, summary
    FROM "{DATABASE}"."orders$snapshots"
    ORDER BY committed_at
""")
print(snapshots.to_string(index=False))
print(f'Snapshots: {len(snapshots)}')


        snapshot_id                committed_at operation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     summary
1871598465773030650 2026-10-16 21:39:46.620 UTC    append                                                             {engine-version=3.5.4-amzn-0, added-data-files=372, total-equality-deletes=0, app-id=spark-application-1792186721723, added-records=2133, total-records=2133, spark.app.id=spark-application-1792186721723, changed-partition-count=372, engine-name=spark, total-position-deletes=0, added-files-size=14906

In [31]:
history = run_query(f"""
    SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor
    FROM "{DATABASE}"."orders$history"
    ORDER BY made_current_at
""")
print(history.to_string(index=False))


            made_current_at         snapshot_id           parent_id is_current_ancestor
2026-10-16 21:39:46.620 UTC 1871598465773030650                None                true
2026-10-16 22:37:59.368 UTC 7756045344025059327 1871598465773030650                true


## Step 12: Time-Travel Query

Iceberg's time travel lets you query a table **as it existed at a previous point in time**
using `FOR TIMESTAMP AS OF`. The bronze `orders` table changed between its two snapshots:
the first CDC run loaded the base data, and the supplemental CDC load (Step 10b) applied
3 updates and 2 deletes.

The approach:
1. Query the current orders row count
2. Get the first snapshot's timestamp from `orders$snapshots`
3. The supplemental CDC load (Step 10b) already created the second snapshot
4. Query the current state (fewer rows because of the deletes)
5. Use `FOR TIMESTAMP AS OF` with the first snapshot's timestamp to read the pre-supplemental state

In [32]:
# 1. Current orders row count
current = run_query(f"""
    SELECT COUNT(*) AS row_count, ROUND(SUM(total_amount), 2) AS total_revenue
    FROM "{DATABASE}"."orders"
""")

# 2. Timestamp of the first snapshot (the state before the supplemental CDC load)
snap_ts = run_query(f'SELECT MIN(committed_at) AS ts FROM "{DATABASE}"."orders$snapshots"').iloc[0]['ts']
print('Time-travel timestamp (first snapshot committed_at):', snap_ts)

# 3. The supplemental CDC load in Step 10b already created the second snapshot

# 4. Current state
print('Current:   ', current.iloc[0].to_dict())

# 5. Time-travel query: orders as of the first snapshot
historical = run_query(f"""
    SELECT COUNT(*) AS row_count, ROUND(SUM(total_amount), 2) AS total_revenue
    FROM "{DATABASE}"."orders"
    FOR TIMESTAMP AS OF TIMESTAMP '{snap_ts}'
""")
print('Historical:', historical.iloc[0].to_dict())

comparison = pd.DataFrame({
    'version': ['Historical (first snapshot)', 'Current'],
    'row_count': [int(historical['row_count'][0]), int(current['row_count'][0])],
    'total_revenue': [float(historical['total_revenue'][0]), float(current['total_revenue'][0])],
})
print()
print(comparison.to_string(index=False))
print(f"\nRow count difference: {comparison['row_count'][0] - comparison['row_count'][1]}")


Time-travel timestamp (first snapshot committed_at): 2026-10-16 21:39:46.620 UTC
Current:    {'row_count': '2131', 'total_revenue': '555744.83'}
Historical: {'row_count': '2133', 'total_revenue': '556462.16'}

                    version  row_count  total_revenue
Historical (first snapshot)       2133      556462.16
                    Current       2131      555744.83

Row count difference: 2


---
## Step 13: Medallion Architecture Summary

Write a summary cell that queries all three layers and prints row counts.
Bronze uses the default catalog; Silver and Gold use `S3T_CATALOG`.

In [33]:
layers = {
    'Bronze': {'tables': ['orders', 'users', 'products', 'events'], 'kwargs': {'database': DATABASE}},
    'Silver': {'tables': ['order_details', 'enriched_events', 'product_performance'],
               'kwargs': {'catalog': S3T_CATALOG, 'database': SILVER_DB}},
    'Gold':   {'tables': ['customer_analytics', 'realtime_metrics'],
               'kwargs': {'catalog': S3T_CATALOG, 'database': GOLD_DB}},
}

print('=' * 52)
print('Medallion Architecture Summary')
print('=' * 52)
for layer, cfg in layers.items():
    print(f'\n{layer} layer')
    for table in cfg['tables']:
        count = run_query(f'SELECT COUNT(*) AS row_count FROM {table}', **cfg['kwargs'])['row_count'][0]
        print(f'  {table:<22} {int(count):>8,} rows')
print('\n' + '=' * 52)


Medallion Architecture Summary

Bronze layer
  orders                    2,131 rows
  users                       912 rows
  products                    391 rows
  events                      498 rows

Silver layer
  order_details             2,024 rows
  enriched_events             498 rows
  product_performance         391 rows

Gold layer
  customer_analytics           13 rows
  realtime_metrics            491 rows

